# LeetCode 540 — Single Element in a Sorted Array bằng Python

Link đề: https://leetcode.com/problems/single-element-in-a-sorted-array/ · Độ khó: **Medium** · Chủ đề: Array, Binary Search

Skeleton LeetCode:

```python
class Solution:
    def singleNonDuplicate(self, nums: List[int]) -> int:
        ...
```

Ghi chú: đề yêu cầu thời gian `O(log n)` và bộ nhớ `O(1)`. Notebook này đi từ cách "đếm" dễ nghĩ nhất, qua các cách `O(n)`, rồi tới binary search (tìm kiếm nhị phân) trên **tính chẵn/lẻ của chỉ số** — cách duy nhất đạt yêu cầu.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mảng đã **sắp xếp**, mọi giá trị xuất hiện **đúng 2 lần** trừ một giá trị xuất hiện **1 lần**; phải tìm giá trị lẻ loi đó trong `O(log n)` thời gian, `O(1)` bộ nhớ.
2. Nắm các công cụ Python/bit: chỉ số chẵn/lẻ (`i % 2`, `i & 1`), phép XOR `^` (đặc biệt `i ^ 1` là "bạn cặp" của vị trí `i`), `collections.Counter`, `range(0, n, 2)`, dịch bit `>> 1`.
3. Đi từ **đếm bằng Counter** → **duyệt theo từng cặp** → **XOR toàn mảng** → **binary search trên chỉ số cặp** → **binary search gọn bằng `mid ^ 1`**.
4. Hiểu mấu chốt: **trước** phần tử lẻ loi, các cặp nằm ở vị trí `(chẵn, lẻ)`; **từ** phần tử lẻ loi trở đi, các cặp bị lệch thành `(lẻ, chẵn)`. Tính chất "đúng/sai" này đơn điệu nên binary search được.
5. Tránh các lỗi kinh điển: dùng `r = mid - 1` làm mất đáp án, so `nums[mid]` với `nums[mid + 1]` mà quên xét chẵn/lẻ, vòng lặp vô hạn do `while l <= r` kèm `r = mid`, và trả về chỉ số thay vì giá trị.

## 2. Đề bài & ví dụ

Cho một mảng số nguyên `nums` đã **sắp xếp tăng dần**, trong đó **mọi phần tử xuất hiện đúng hai lần, trừ một phần tử chỉ xuất hiện đúng một lần**. Hãy trả về phần tử chỉ xuất hiện một lần đó.

Lời giải phải chạy trong `O(log n)` thời gian và `O(1)` bộ nhớ.

**Ví dụ 1**

- Input: `nums = [1,1,2,3,3,4,4,8,8]`
- Output: `2`

**Ví dụ 2**

- Input: `nums = [3,3,7,7,10,11,11]`
- Output: `10`

**Ràng buộc:**

- `1 <= nums.length <= 10^5`
- `0 <= nums[i] <= 10^5`

Hệ quả quan trọng: vì các giá trị còn lại đi theo cặp nên **độ dài mảng luôn là số lẻ** (`2k + 1`). Mảng chỉ có 1 phần tử thì đáp án là chính nó.

In [1]:
from typing import List
from collections import Counter
from functools import reduce
from operator import xor
import random

examples = [
    ([1, 1, 2, 3, 3, 4, 4, 8, 8], 2),
    ([3, 3, 7, 7, 10, 11, 11], 10),
]
for nums, expected in examples:
    print("n =", len(nums), "| lẻ?", len(nums) % 2 == 1, "| kỳ vọng:", expected)

n = 9 | lẻ? True | kỳ vọng: 2
n = 7 | lẻ? True | kỳ vọng: 10


## 3. Kiến thức nền

### 3.1. Chỉ số (index) chẵn và lẻ

Phần tử đầu tiên có chỉ số `0` (chẵn), phần tử thứ hai có chỉ số `1` (lẻ), cứ xen kẽ như vậy. Ta kiểm tra bằng `i % 2` (phần dư khi chia 2: `0` là chẵn, `1` là lẻ) hoặc `i & 1` (lấy bit thấp nhất, cho cùng kết quả nhưng nhanh hơn một chút).

In [2]:
for i in range(6):
    print(i, "chẵn" if i % 2 == 0 else "lẻ", "| i % 2 =", i % 2, "| i & 1 =", i & 1)

0 chẵn | i % 2 = 0 | i & 1 = 0
1 lẻ | i % 2 = 1 | i & 1 = 1
2 chẵn | i % 2 = 0 | i & 1 = 0
3 lẻ | i % 2 = 1 | i & 1 = 1
4 chẵn | i % 2 = 0 | i & 1 = 0
5 lẻ | i % 2 = 1 | i & 1 = 1


### 3.2. Phép XOR `^` và "bạn cặp" `i ^ 1`

`^` là phép **XOR** (loại trừ) từng bit: hai bit giống nhau cho `0`, khác nhau cho `1`. Ví dụ `5 ^ 3`: `101 ^ 011 = 110 = 6`.

Hai tính chất XOR cần nhớ:

- `a ^ a = 0` (một số XOR với chính nó triệt tiêu), và `a ^ 0 = a`.
- XOR có tính giao hoán, kết hợp nên **thứ tự không quan trọng**.

Trường hợp đặc biệt rất hay dùng: **`i ^ 1` lật bit thấp nhất của `i`**. Nghĩa là `0 <-> 1`, `2 <-> 3`, `4 <-> 5`, ... Với chỉ số chẵn, `i ^ 1 = i + 1`; với chỉ số lẻ, `i ^ 1 = i - 1`. Tức `i ^ 1` luôn là "bạn cặp" của `i` nếu ta ghép các chỉ số thành các cặp `(0,1), (2,3), (4,5), ...`.

In [3]:
print([i ^ 1 for i in range(10)])           # [1, 0, 3, 2, 5, 4, 7, 6, 9, 8]
print("5 ^ 3 =", 5 ^ 3, "| 7 ^ 7 =", 7 ^ 7, "| 7 ^ 0 =", 7 ^ 0)
for i in range(6):
    side = "i + 1" if i % 2 == 0 else "i - 1"
    print(f"i={i}: i ^ 1 = {i ^ 1}  (tức {side})")

[1, 0, 3, 2, 5, 4, 7, 6, 9, 8]
5 ^ 3 = 6 | 7 ^ 7 = 0 | 7 ^ 0 = 7
i=0: i ^ 1 = 1  (tức i + 1)
i=1: i ^ 1 = 0  (tức i - 1)
i=2: i ^ 1 = 3  (tức i + 1)
i=3: i ^ 1 = 2  (tức i - 1)
i=4: i ^ 1 = 5  (tức i + 1)
i=5: i ^ 1 = 4  (tức i - 1)


### 3.3. Khuôn mẫu binary search `while l < r`

Có nhiều kiểu viết binary search. Với bài tìm **vị trí đầu tiên thỏa một điều kiện** (điều kiện sai ... sai rồi đúng ... đúng), kiểu gọn nhất là:

```
l, r = 0, n - 1            # đáp án chắc chắn nằm trong [l, r]
while l < r:
    mid = (l + r) >> 1     # chia đôi, làm tròn xuống
    if điều_kiện(mid):     # mid có thể chính là đáp án
        r = mid            # giữ mid lại
    else:
        l = mid + 1        # mid chắc chắn sai, bỏ mid
return l                   # l == r là vị trí đầu tiên đúng
```

`(l + r) >> 1` bằng `(l + r) // 2` (dịch phải 1 bit = chia đôi làm tròn xuống). Chú ý `mid < r` luôn đúng khi `l < r`, vì `mid` làm tròn xuống. Ta sẽ cần điều này để chứng minh `mid ^ 1` không vượt khỏi mảng.

In [4]:
# Ví dụ nhỏ: tìm vị trí đầu tiên có giá trị >= 5 trong mảng tăng dần
a = [1, 3, 4, 5, 7, 9]
l, r = 0, len(a) - 1
while l < r:
    mid = (l + r) >> 1
    print(f"l={l} r={r} mid={mid} a[mid]={a[mid]}", "-> r = mid" if a[mid] >= 5 else "-> l = mid + 1")
    if a[mid] >= 5:
        r = mid
    else:
        l = mid + 1
print("vị trí đầu tiên >= 5:", l, "giá trị", a[l])

l=0 r=5 mid=2 a[mid]=4 -> l = mid + 1
l=3 r=5 mid=4 a[mid]=7 -> r = mid
l=3 r=4 mid=3 a[mid]=5 -> r = mid
vị trí đầu tiên >= 5: 3 giá trị 5


### 3.4. `Counter`, `range` có bước nhảy, `reduce` với XOR

- `Counter(nums)` đếm số lần xuất hiện của từng giá trị, trả về dict-like: `Counter([1,1,2]) -> {1: 2, 2: 1}`.
- `range(0, n, 2)` đi `0, 2, 4, ...` (bước nhảy 2) — hợp để duyệt theo **cặp**.
- `reduce(xor, nums)` XOR liên tiếp mọi phần tử: `((nums[0] ^ nums[1]) ^ nums[2]) ^ ...`.

In [5]:
nums = [1, 1, 2, 3, 3, 4, 4, 8, 8]
c = Counter(nums)
print(c)
print([v for v, cnt in c.items() if cnt == 1])
print(list(range(0, 9, 2)))
print(reduce(xor, nums))     # các cặp triệt tiêu, chỉ còn 2

Counter({1: 2, 3: 2, 4: 2, 8: 2, 2: 1})
[2]
[0, 2, 4, 6, 8]
2


## 4. Ý tưởng nền tảng / trực giác

Hãy viết các phần tử cùng chỉ số của chúng và để ý **cặp nào đi với nhau**. Với `[1,1,2,3,3,4,4,8,8]`:

```
chỉ số:   0  1  2  3  4  5  6  7  8
giá trị:  1  1  2  3  3  4  4  8  8
          |__|  ^  |__|  |__|  |__|
          cặp   lẻ   cặp    cặp   cặp
         (0,1)  loi (3,4)  (5,6)  (7,8)
```

- **Trước** phần tử lẻ loi (chỉ số 2): cặp `(0,1)` có dạng `(chẵn, lẻ)`.
- Phần tử lẻ loi ở chỉ số **chẵn** (2). Sau nó mọi thứ bị **đẩy lệch 1 vị trí**: cặp `(3,4)`, `(5,6)`, `(7,8)` có dạng `(lẻ, chẵn)`.

Lý do: trước phần tử lẻ loi có một số **chẵn** phần tử (toàn là các cặp đầy đủ), nên phần tử lẻ loi luôn nằm ở chỉ số chẵn. Ngay sau đó, mỗi cặp bắt đầu ở chỉ số lẻ.

Đây chính là "dấu hiệu" để binary search: nhìn một chỉ số `mid`, xem nó đang **nằm trong cặp kiểu (chẵn, lẻ)** hay **cặp kiểu (lẻ, chẵn)** là biết ta đang đứng bên trái hay bên phải phần tử lẻ loi.

In [6]:
def show_pair_side(nums):
    """Với mỗi chỉ số: bạn cặp (giá trị bằng) nằm bên trái, bên phải hay không có."""
    for i, v in enumerate(nums):
        left_same = i > 0 and nums[i - 1] == v
        right_same = i + 1 < len(nums) and nums[i + 1] == v
        who = "bạn cặp bên TRÁI" if left_same else ("bạn cặp bên PHẢI" if right_same else "LẺ LOI")
        parity = "chẵn" if i % 2 == 0 else "lẻ"
        print(f"i={i} ({parity}) giá trị {v}: {who}")

show_pair_side([1, 1, 2, 3, 3, 4, 4, 8, 8])

i=0 (chẵn) giá trị 1: bạn cặp bên PHẢI
i=1 (lẻ) giá trị 1: bạn cặp bên TRÁI
i=2 (chẵn) giá trị 2: LẺ LOI
i=3 (lẻ) giá trị 3: bạn cặp bên PHẢI
i=4 (chẵn) giá trị 3: bạn cặp bên TRÁI
i=5 (lẻ) giá trị 4: bạn cặp bên PHẢI
i=6 (chẵn) giá trị 4: bạn cặp bên TRÁI
i=7 (lẻ) giá trị 8: bạn cặp bên PHẢI
i=8 (chẵn) giá trị 8: bạn cặp bên TRÁI


Đọc kết quả: ở bên trái phần tử lẻ loi, chỉ số **chẵn** có bạn cặp bên **phải**, chỉ số **lẻ** có bạn cặp bên trái. Ở bên phải phần tử lẻ loi thì **ngược lại**: chỉ số chẵn có bạn cặp bên trái, chỉ số lẻ có bạn cặp bên phải.

Nói bằng `i ^ 1`: bên trái phần tử lẻ loi, `nums[i] == nums[i ^ 1]` (bạn cặp đúng là `i ^ 1`); từ phần tử lẻ loi trở đi, `nums[i] != nums[i ^ 1]`. Ta sẽ chứng minh chặt ở mục 10.

## 5. Cách 1 — Brute Force: đếm bằng Counter

### Ý tưởng

Đề không cấm ta bỏ qua việc mảng đã sắp xếp. Cách dễ nghĩ nhất: đếm số lần xuất hiện của từng giá trị, rồi lấy giá trị có số đếm bằng `1`.

### Code

In [7]:
def single_counter(nums):
    cnt = Counter(nums)
    for value, times in cnt.items():
        if times == 1:
            return value

for nums, expected in examples:
    print(nums, "->", single_counter(nums), "(kỳ vọng", expected, ")")

[1, 1, 2, 3, 3, 4, 4, 8, 8] -> 2 (kỳ vọng 2 )
[3, 3, 7, 7, 10, 11, 11] -> 10 (kỳ vọng 10 )


### Độ phức tạp và nút thắt

- Time `O(n)`: duyệt mảng một lần để đếm, rồi duyệt bảng đếm.
- Space `O(n)`: bảng đếm có tới `(n + 1) / 2` giá trị khác nhau.

**Nút thắt:** vi phạm cả hai yêu cầu của đề (`O(n)` thời gian thay vì `O(log n)`, `O(n)` bộ nhớ thay vì `O(1)`). Vấn đề là ta chưa dùng thông tin "mảng đã sắp xếp" — thông tin đó chính là thứ cho phép chia đôi.

## 6. Ý tưởng thuật toán / bước đột phá

Để dùng được binary search ta cần một **vị từ (predicate)** `P(i)` có tính chất: **sai** với mọi `i` bên trái đáp án, **đúng** với mọi `i` từ đáp án trở đi. Khi đó đáp án là **chỉ số đúng đầu tiên**, và ta tìm bằng khuôn mẫu ở mục 3.3.

Từ trực giác mục 4, gọi `s` là chỉ số phần tử lẻ loi, ta chọn:

> `P(i)`: `nums[i] != nums[i ^ 1]` ("`i` không còn nằm trong cặp kiểu `(chẵn, lẻ)` nguyên vẹn")

- `i < s`: `i` và `i ^ 1` là hai phần tử của một cặp `(chẵn, lẻ)` → bằng nhau → `P(i)` **sai**.
- `i = s`: `i` chẵn, `i ^ 1 = i + 1` thuộc cặp tiếp theo (giá trị khác) → `P(i)` **đúng**.
- `i > s`: các cặp lệch thành `(lẻ, chẵn)`, nên `i ^ 1` luôn rơi sang cặp bên cạnh, giá trị khác → `P(i)` **đúng**.

Vậy `P` có dạng `sai ... sai đúng ... đúng`, và chỉ số đúng đầu tiên chính là `s`.

In [8]:
def predicate_table(nums):
    n = len(nums)
    out = []
    for i in range(n):
        j = i ^ 1
        if j >= n:                 # chỉ xảy ra với i = n - 1 (chẵn): coi như "đúng"
            out.append((i, nums[i], None, True))
        else:
            out.append((i, nums[i], nums[j], nums[i] != nums[j]))
    return out

print("i | nums[i] | nums[i^1] | P(i)")
for i, v, w, p in predicate_table([1, 1, 2, 3, 3, 4, 4, 8, 8]):
    print(i, "|", v, "|", w, "|", p)

i | nums[i] | nums[i^1] | P(i)
0 | 1 | 1 | False
1 | 1 | 1 | False
2 | 2 | 3 | True
3 | 3 | 2 | True
4 | 3 | 4 | True
5 | 4 | 3 | True
6 | 4 | 8 | True
7 | 8 | 4 | True
8 | 8 | None | True


Bảng cho thấy `P` đúng dạng `F F T T T T T T T`. Đây là bước đột phá: **bài toán "tìm giá trị" biến thành "tìm vị trí đầu tiên đúng của một vị từ đơn điệu"**, và việc kiểm tra `P(mid)` chỉ tốn `O(1)`.

Ghi chú: với `i = n - 1` (chỉ số chẵn vì `n` lẻ), `i ^ 1 = n` nằm ngoài mảng. Ta coi `P(n - 1)` là đúng (đây là "lính canh" giúp đáp án luôn tồn tại), và ta sẽ thấy binary search không bao giờ phải gọi `P(n - 1)` vì `mid < r <= n - 1`.

## 7. Các cách tiếp theo

Phần này có thêm 4 cách nữa (tổng 5 cách): duyệt theo cặp `O(n)`, XOR toàn mảng `O(n)`, binary search trên chỉ số cặp, và binary search `mid ^ 1`.

### 7.1. Cách 2 — Duyệt theo từng cặp (tận dụng sắp xếp)

**Giải thích từng đoạn.** Vì mảng đã sắp xếp và các giá trị đi theo cặp liền nhau, ta nhìn mảng theo từng cặp `(nums[i], nums[i+1])` với `i = 0, 2, 4, ...`. Nếu `nums[i] != nums[i + 1]` thì `nums[i]` lẻ loi (vì nếu `nums[i]` có bạn cặp thì bạn phải nằm ngay cạnh). Nếu duyệt hết mà không thấy cặp nào lệch, phần tử lẻ loi là phần tử **cuối cùng**.

In [9]:
def single_pairs(nums):
    n = len(nums)
    for i in range(0, n - 1, 2):          # i = 0, 2, 4, ... (dừng trước n - 1)
        if nums[i] != nums[i + 1]:
            return nums[i]
    return nums[-1]                       # mọi cặp đều khớp -> phần tử cuối lẻ loi

for nums, expected in examples:
    print(nums, "->", single_pairs(nums))
print(single_pairs([5]), single_pairs([1, 1, 2]), single_pairs([1, 2, 2]))

[1, 1, 2, 3, 3, 4, 4, 8, 8] -> 2
[3, 3, 7, 7, 10, 11, 11] -> 10
5 2 1


In [10]:
class Solution:
    def singleNonDuplicate(self, nums: List[int]) -> int:
        n = len(nums)
        for i in range(0, n - 1, 2):
            if nums[i] != nums[i + 1]:
                return nums[i]
        return nums[-1]

print(Solution().singleNonDuplicate([1, 1, 2, 3, 3, 4, 4, 8, 8]))

2


**Độ phức tạp:** Time `O(n)` (tối đa `n / 2` lần so sánh), Space `O(1)`. Đã đạt `O(1)` bộ nhớ nhưng thời gian vẫn tuyến tính.

### 7.2. Cách 3 — XOR toàn mảng (không cần sắp xếp)

**Giải thích từng đoạn.** Mỗi giá trị xuất hiện 2 lần sẽ tự triệt tiêu khi XOR (`a ^ a = 0`), `0 ^ x = x`. Vì XOR giao hoán và kết hợp, ta XOR cả mảng theo thứ tự bất kỳ thì chỉ còn phần tử lẻ loi. Cách này **không dùng tính sắp xếp** nên cũng đúng với mảng chưa sắp xếp (đó là bài *Single Number*, LeetCode 136).

In [11]:
def single_xor(nums):
    ans = 0
    for x in nums:
        ans ^= x            # ans = ans ^ x
    return ans

for nums, expected in examples:
    print(nums, "->", single_xor(nums))
print(single_xor([4, 1, 2, 1, 2]))   # không cần sắp xếp vẫn đúng

[1, 1, 2, 3, 3, 4, 4, 8, 8] -> 2
[3, 3, 7, 7, 10, 11, 11] -> 10
4


In [12]:
class Solution:
    def singleNonDuplicate(self, nums: List[int]) -> int:
        ans = 0
        for x in nums:
            ans ^= x
        return ans

print(Solution().singleNonDuplicate([3, 3, 7, 7, 10, 11, 11]))

10


**Độ phức tạp:** Time `O(n)`, Space `O(1)`. Rất ngắn gọn, nhưng vẫn là `O(n)` — đề muốn `O(log n)` nên vẫn chưa đạt.

### 7.3. Cách 4 — Binary search trên chỉ số *cặp*

**Giải thích từng đoạn.** Ghép mảng thành các cặp `(2k, 2k+1)` với `k = 0 .. (n-1)/2`; cặp cuối cùng chỉ có 1 phần tử nếu lẻ loi nằm cuối. Gọi `k*` là cặp chứa phần tử lẻ loi (nó luôn nằm ở đầu cặp, chỉ số `2k*`). Với mỗi `k`:

- `k < k*`: cặp nguyên vẹn, `nums[2k] == nums[2k+1]`.
- `k = k*`: `nums[2k] != nums[2k+1]` (hoặc `2k` là chỉ số cuối, không có `2k+1`).
- `k > k*`: cặp đã lệch, `nums[2k] != nums[2k+1]` vì `nums[2k]` là bạn cặp của `nums[2k-1]`.

Vậy ta tìm `k` **nhỏ nhất** sao cho `2k == n - 1` hoặc `nums[2k] != nums[2k+1]`. Binary search trên `k` thuộc `[0, (n - 1) / 2]`; đáp án là `nums[2k]`.

In [13]:
def single_pair_index(nums):
    n = len(nums)
    lo, hi = 0, (n - 1) // 2               # k nằm trong [0, (n-1)/2]; cặp cuối có thể chỉ có 1 phần tử
    while lo < hi:
        k = (lo + hi) >> 1                 # k < hi nên 2k + 1 <= n - 1: luôn hợp lệ
        if nums[2 * k] != nums[2 * k + 1]:
            hi = k                         # cặp k đã lệch: lẻ loi ở cặp k hoặc trước đó
        else:
            lo = k + 1                     # cặp k còn nguyên: lẻ loi ở bên phải
    return nums[2 * lo]

for nums, expected in examples:
    print(nums, "->", single_pair_index(nums))
print(single_pair_index([5]), single_pair_index([1, 1, 2]), single_pair_index([1, 2, 2]))

[1, 1, 2, 3, 3, 4, 4, 8, 8] -> 2
[3, 3, 7, 7, 10, 11, 11] -> 10
5 2 1


In [14]:
class Solution:
    def singleNonDuplicate(self, nums: List[int]) -> int:
        lo, hi = 0, (len(nums) - 1) // 2
        while lo < hi:
            k = (lo + hi) >> 1
            if nums[2 * k] != nums[2 * k + 1]:
                hi = k
            else:
                lo = k + 1
        return nums[2 * lo]

print(Solution().singleNonDuplicate([1, 1, 2, 3, 3, 4, 4, 8, 8]))

2


**Độ phức tạp:** Time `O(log n)` (khoảng `log2(n / 2)` bước), Space `O(1)`. Đạt yêu cầu. Nhược điểm: phải nghĩ bằng "chỉ số cặp" và đổi qua lại `2k`, `2k+1`; cách sau gọn hơn.

### 7.4. Cách 5 — Binary search gọn với `mid ^ 1` (lời giải nộp)

**Giải thích từng đoạn.** Dùng thẳng vị từ ở mục 6 trên **chỉ số của mảng**, không cần đổi sang chỉ số cặp:

1. `l, r = 0, n - 1`: đáp án nằm trong `[l, r]`.
2. `mid = (l + r) >> 1`. So `nums[mid]` với bạn cặp **giả định** `nums[mid ^ 1]`.
3. Nếu khác nhau: `P(mid)` đúng → đáp án ở `mid` hoặc bên trái → `r = mid`.
4. Nếu bằng nhau: `P(mid)` sai → đáp án nằm bên phải → `l = mid + 1`.
5. Khi `l == r`, trả về `nums[l]`.

In [15]:
def single_bs_xor(nums):
    l, r = 0, len(nums) - 1
    while l < r:
        mid = (l + r) >> 1
        if nums[mid] != nums[mid ^ 1]:
            r = mid
        else:
            l = mid + 1
    return nums[l]

for nums, expected in examples:
    print(nums, "->", single_bs_xor(nums))
print(single_bs_xor([5]), single_bs_xor([1, 1, 2]), single_bs_xor([1, 2, 2]))

[1, 1, 2, 3, 3, 4, 4, 8, 8] -> 2
[3, 3, 7, 7, 10, 11, 11] -> 10
5 2 1


In [16]:
class Solution:
    def singleNonDuplicate(self, nums: List[int]) -> int:
        l, r = 0, len(nums) - 1
        while l < r:
            mid = (l + r) >> 1
            if nums[mid] != nums[mid ^ 1]:
                r = mid
            else:
                l = mid + 1
        return nums[l]

print(Solution().singleNonDuplicate([3, 3, 7, 7, 10, 11, 11]))

10


**Độ phức tạp:** Time `O(log n)`, Space `O(1)`. Đây là cách ngắn nhất đạt đúng yêu cầu đề.

### 7.5. Một biến thể thủ công của cách 5 (không dùng XOR)

Nếu chưa quen `^`, ta có thể viết điều kiện chẵn/lẻ tường minh: `mid` chẵn thì so với `mid + 1`, `mid` lẻ thì so với `mid - 1`. Logic y hệt, chỉ dài hơn.

In [17]:
def single_bs_manual(nums):
    l, r = 0, len(nums) - 1
    while l < r:
        mid = (l + r) >> 1
        partner = mid + 1 if mid % 2 == 0 else mid - 1
        if nums[mid] != nums[partner]:
            r = mid
        else:
            l = mid + 1
    return nums[l]

for nums, expected in examples:
    print(nums, "->", single_bs_manual(nums))

[1, 1, 2, 3, 3, 4, 4, 8, 8] -> 2
[3, 3, 7, 7, 10, 11, 11] -> 10


## 8. TRACE từng bước

Trace ví dụ 1: `nums = [1,1,2,3,3,4,4,8,8]`, `n = 9`, kỳ vọng `2`.

| bước | l | r | mid | mid ^ 1 | nums[mid] | nums[mid^1] | so sánh | hành động |
|---|---|---|---|---|---|---|---|---|
| 1 | 0 | 8 | 4 | 5 | 3 | 4 | khác | r = 4 |
| 2 | 0 | 4 | 2 | 3 | 2 | 3 | khác | r = 2 |
| 3 | 0 | 2 | 1 | 0 | 1 | 1 | bằng | l = 2 |
| dừng | 2 | 2 | | | | | | trả `nums[2] = 2` |

Nhận xét: ở bước 3, `mid = 1` (lẻ) nên bạn cặp là `mid ^ 1 = 0`, và `nums[1] == nums[0]` → cặp còn nguyên → phần tử lẻ loi ở bên phải `mid`.

In [18]:
def trace_single(nums):
    l, r = 0, len(nums) - 1
    print("nums =", nums)
    step = 0
    while l < r:
        step += 1
        mid = (l + r) >> 1
        j = mid ^ 1
        same = nums[mid] == nums[j]
        action = f"l = {mid + 1}" if same else f"r = {mid}"
        print(f"bước {step}: l={l} r={r} mid={mid} mid^1={j} nums[mid]={nums[mid]} nums[mid^1]={nums[j]} "
              f"-> {'bằng' if same else 'khác'} -> {action}")
        if same:
            l = mid + 1
        else:
            r = mid
    print(f"dừng: l=r={l} -> đáp án {nums[l]}")
    return nums[l]

assert trace_single([1, 1, 2, 3, 3, 4, 4, 8, 8]) == 2

nums = [1, 1, 2, 3, 3, 4, 4, 8, 8]
bước 1: l=0 r=8 mid=4 mid^1=5 nums[mid]=3 nums[mid^1]=4 -> khác -> r = 4
bước 2: l=0 r=4 mid=2 mid^1=3 nums[mid]=2 nums[mid^1]=3 -> khác -> r = 2
bước 3: l=0 r=2 mid=1 mid^1=0 nums[mid]=1 nums[mid^1]=1 -> bằng -> l = 2
dừng: l=r=2 -> đáp án 2


Trace ví dụ 2: `nums = [3,3,7,7,10,11,11]`, `n = 7`, kỳ vọng `10`.

| bước | l | r | mid | mid ^ 1 | nums[mid] | nums[mid^1] | so sánh | hành động |
|---|---|---|---|---|---|---|---|---|
| 1 | 0 | 6 | 3 | 2 | 7 | 7 | bằng | l = 4 |
| 2 | 4 | 6 | 5 | 4 | 11 | 10 | khác | r = 5 |
| 3 | 4 | 5 | 4 | 5 | 10 | 11 | khác | r = 4 |
| dừng | 4 | 4 | | | | | | trả `nums[4] = 10` |

In [19]:
assert trace_single([3, 3, 7, 7, 10, 11, 11]) == 10
print()
# Hai trường hợp biên: lẻ loi ở ĐẦU và ở CUỐI mảng
assert trace_single([0, 1, 1, 2, 2]) == 0
print()
assert trace_single([1, 1, 2, 2, 9]) == 9

nums = [3, 3, 7, 7, 10, 11, 11]
bước 1: l=0 r=6 mid=3 mid^1=2 nums[mid]=7 nums[mid^1]=7 -> bằng -> l = 4
bước 2: l=4 r=6 mid=5 mid^1=4 nums[mid]=11 nums[mid^1]=10 -> khác -> r = 5
bước 3: l=4 r=5 mid=4 mid^1=5 nums[mid]=10 nums[mid^1]=11 -> khác -> r = 4
dừng: l=r=4 -> đáp án 10

nums = [0, 1, 1, 2, 2]
bước 1: l=0 r=4 mid=2 mid^1=3 nums[mid]=1 nums[mid^1]=2 -> khác -> r = 2
bước 2: l=0 r=2 mid=1 mid^1=0 nums[mid]=1 nums[mid^1]=0 -> khác -> r = 1
bước 3: l=0 r=1 mid=0 mid^1=1 nums[mid]=0 nums[mid^1]=1 -> khác -> r = 0
dừng: l=r=0 -> đáp án 0

nums = [1, 1, 2, 2, 9]
bước 1: l=0 r=4 mid=2 mid^1=3 nums[mid]=2 nums[mid^1]=2 -> bằng -> l = 3
bước 2: l=3 r=4 mid=3 mid^1=2 nums[mid]=2 nums[mid^1]=2 -> bằng -> l = 4
dừng: l=r=4 -> đáp án 9


Quan sát hai trace biên: khi lẻ loi nằm **đầu** mảng, `P(mid)` luôn đúng nên `r` cứ co về `0`. Khi lẻ loi nằm **cuối** mảng, `P(mid)` luôn sai (với mọi `mid < n - 1`) nên `l` cứ dời tới `n - 1` — đúng như "lính canh" đã nói ở mục 6.

## 9. Code nộp LeetCode

Lời giải đúng yêu cầu `O(log n)` / `O(1)` (Cách 5):

In [20]:
class Solution:
    def singleNonDuplicate(self, nums: List[int]) -> int:
        l, r = 0, len(nums) - 1
        while l < r:
            mid = (l + r) >> 1
            if nums[mid] != nums[mid ^ 1]:
                r = mid
            else:
                l = mid + 1
        return nums[l]


sol = Solution()
for nums, expected in examples:
    got = sol.singleNonDuplicate(nums)
    print(nums, "->", got)
    assert got == expected

[1, 1, 2, 3, 3, 4, 4, 8, 8] -> 2
[3, 3, 7, 7, 10, 11, 11] -> 10


## 10. Vì sao đúng

**Khẳng định.** Gọi `s` là chỉ số của phần tử lẻ loi. Khi đó với mọi `i` thỏa `0 <= i <= n - 2`:

- `i < s` thì `nums[i] == nums[i ^ 1]`;
- `i >= s` thì `nums[i] != nums[i ^ 1]`.

**Chứng minh.**

1. Trước `s` có `s` phần tử, tất cả đi thành các cặp nguyên vẹn nên `s` **chẵn**. Các cặp đó chiếm các chỉ số `(0,1), (2,3), ..., (s-2, s-1)`, đúng bằng các cặp `(i, i ^ 1)`. Do đó với `i < s`, `nums[i] == nums[i ^ 1]`.
2. Tại `i = s` (chẵn): `i ^ 1 = s + 1`. Phần tử `nums[s + 1]` thuộc một cặp khác (vì `nums[s]` chỉ xuất hiện một lần) và mảng sắp xếp tăng nên `nums[s + 1] > nums[s]`. Vậy `nums[s] != nums[s ^ 1]`. (Nếu `s = n - 1` thì `s ^ 1 = n` ra ngoài mảng; ta chỉ xét `i <= n - 2` nên trường hợp này không xảy ra trong vòng lặp.)
3. Với `i > s`: các cặp lệch kiểu `(s+1, s+2), (s+3, s+4), ...` tức `(lẻ, chẵn)`. Với `i` lẻ, bạn cặp thật là `i + 1` còn `i ^ 1 = i - 1` thuộc cặp trước đó; với `i` chẵn, bạn cặp thật là `i - 1` còn `i ^ 1 = i + 1` thuộc cặp sau. Trong cả hai trường hợp `i ^ 1` rơi vào cặp khác nên giá trị khác (mảng sắp xếp, hai cặp liên tiếp có giá trị khác nhau) → `nums[i] != nums[i ^ 1]`.

**Bất biến của vòng lặp:** `s` luôn nằm trong `[l, r]`.

- Ban đầu `[0, n - 1]` chứa `s`.
- Nếu `nums[mid] != nums[mid ^ 1]` thì `mid >= s`, ta đặt `r = mid`; `s <= mid` nên vẫn trong `[l, mid]`.
- Nếu `nums[mid] == nums[mid ^ 1]` thì `mid < s`, đặt `l = mid + 1 <= s`.
- Mỗi bước, khoảng `[l, r]` thu hẹp (vì `mid < r` nên `r = mid` làm `r` giảm; `l = mid + 1` làm `l` tăng), nên vòng lặp kết thúc với `l == r == s`.

**`mid ^ 1` không vượt mảng:** trong vòng lặp `l < r` nên `mid < r <= n - 1`, tức `mid <= n - 2`. Nếu `mid` chẵn thì `mid ^ 1 = mid + 1 <= n - 1`, hợp lệ; nếu `mid` lẻ thì `mid ^ 1 = mid - 1 >= 0`, hợp lệ.

In [21]:
# Kiểm chứng thực nghiệm mệnh đề ở mục 10 với mọi vị trí s và nhiều độ dài
def build_with_single_at(k_pairs, s_pair):
    """Mảng có k_pairs cặp + 1 phần tử lẻ loi chèn trước cặp thứ s_pair (0..k_pairs)."""
    arr, v = [], 0
    for idx in range(k_pairs + 1):
        if idx == s_pair:
            arr.append(v); v += 1
        if idx < k_pairs:
            arr += [v, v]; v += 1
    return arr

checked = 0
for k in range(0, 12):
    for sp in range(k + 1):
        arr = build_with_single_at(k, sp)
        s = 2 * sp                                  # lẻ loi luôn ở chỉ số chẵn
        for i in range(len(arr) - 1):
            assert (arr[i] != arr[i ^ 1]) == (i >= s), (arr, i)
            checked += 1
print("đã kiểm", checked, "ô: mệnh đề đúng")

đã kiểm 1144 ô: mệnh đề đúng


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Counter | `O(n)` | `O(n)` |
| 2. Duyệt theo cặp | `O(n)` | `O(1)` |
| 3. XOR toàn mảng | `O(n)` | `O(1)` |
| 4. Binary search chỉ số cặp | `O(log n)` | `O(1)` |
| 5. Binary search `mid ^ 1` (nộp) | `O(log n)` | `O(1)` |

Với cách 5: mỗi vòng lặp giảm độ dài `r - l + 1` xuống khoảng một nửa nên chỉ có `O(log n)` vòng, mỗi vòng `O(1)`; chỉ dùng vài biến `l, r, mid` nên Space `O(1)`. Với `n = 10^5`, tối đa khoảng 17 vòng lặp.

In [22]:
import math
for n in [1, 9, 1001, 99999]:
    print(f"n={n}: tối đa ~{math.ceil(math.log2(n)) if n > 1 else 0} vòng lặp binary search")

# Đếm số vòng thực tế trên mảng lớn nhất
def count_iters(nums):
    l, r, it = 0, len(nums) - 1, 0
    while l < r:
        it += 1
        mid = (l + r) >> 1
        if nums[mid] != nums[mid ^ 1]:
            r = mid
        else:
            l = mid + 1
    return it

big = build_with_single_at(49999, 31234)
print(len(big), "phần tử ->", count_iters(big), "vòng lặp")

n=1: tối đa ~0 vòng lặp binary search
n=9: tối đa ~4 vòng lặp binary search
n=1001: tối đa ~10 vòng lặp binary search
n=99999: tối đa ~17 vòng lặp binary search
99999 phần tử -> 17 vòng lặp


## 12. Bộ test

Gồm: ví dụ đề, biên (độ dài 1, 3; lẻ loi ở đầu / giữa / cuối), phản ví dụ cho các lời giải sai, đối chiếu cả 5 cách (và biến thể thủ công) bằng `assert` trên rất nhiều input, test ngẫu nhiên có seed đúng ràng buộc, và test kích thước lớn nhất.

In [23]:
ALL_METHODS = {
    "counter": single_counter,
    "pairs": single_pairs,
    "xor": single_xor,
    "bs_pair_index": single_pair_index,
    "bs_mid_xor": single_bs_xor,
    "bs_manual": single_bs_manual,
    "leetcode": Solution().singleNonDuplicate,
}

fixed_cases = [
    ([1, 1, 2, 3, 3, 4, 4, 8, 8], 2),   # ví dụ 1
    ([3, 3, 7, 7, 10, 11, 11], 10),     # ví dụ 2
    ([5], 5),                           # độ dài 1
    ([0], 0),                           # giá trị 0
    ([1, 1, 2], 2),                     # lẻ loi cuối, n = 3
    ([1, 2, 2], 1),                     # lẻ loi đầu, n = 3
    ([1, 1, 2, 2, 3], 3),               # lẻ loi cuối
    ([0, 1, 1, 2, 2], 0),               # lẻ loi đầu, có giá trị 0
    ([1, 1, 2, 3, 3], 2),               # lẻ loi giữa
    ([99999, 100000, 100000], 99999),  # biên giá trị lớn
]
for nums, expected in fixed_cases:
    for name, fn in ALL_METHODS.items():
        assert fn(nums) == expected, (name, nums, fn(nums), expected)
print("test cố định:", len(fixed_cases), "bộ x", len(ALL_METHODS), "cách: OK")

test cố định: 10 bộ x 7 cách: OK


In [24]:
# Phản ví dụ: so nums[mid] với nums[mid + 1] mà KHÔNG xét chẵn/lẻ -> sai
def wrong_no_parity(nums):
    l, r = 0, len(nums) - 1
    while l < r:
        mid = (l + r) >> 1
        if nums[mid] != nums[mid + 1]:
            r = mid
        else:
            l = mid + 1
    return nums[l]

print("ví dụ 1 sai:", wrong_no_parity([1, 1, 2, 3, 3, 4, 4, 8, 8]), "(đúng là 2)")
print("ví dụ 2 sai:", wrong_no_parity([3, 3, 7, 7, 10, 11, 11]), "(đúng là 10)")

# Tìm một phản ví dụ nhỏ nhất bằng vét cạn
found = None
for k in range(0, 6):
    for sp in range(k + 1):
        arr = build_with_single_at(k, sp)
        if wrong_no_parity(arr) != single_xor(arr):
            found = arr
            break
    if found:
        break
print("phản ví dụ nhỏ nhất:", found, "| lời giải sai trả", wrong_no_parity(found), "| đúng", single_xor(found))

ví dụ 1 sai: 1 (đúng là 2)
ví dụ 2 sai: 3 (đúng là 10)
phản ví dụ nhỏ nhất: [0, 1, 1] | lời giải sai trả 1 | đúng 0


In [25]:
# Đối chiếu mọi cách trên TOÀN BỘ mảng hợp lệ nhỏ: mọi số cặp k <= 14, mọi vị trí lẻ loi
total = 0
for k in range(0, 15):
    for sp in range(k + 1):
        arr = build_with_single_at(k, sp)
        expected = single_counter(arr)            # oracle độc lập: đếm
        for name, fn in ALL_METHODS.items():
            assert fn(arr) == expected, (name, arr)
        total += 1
print("đối chiếu vét cạn:", total, "mảng, mọi cách đều khớp")

đối chiếu vét cạn: 120 mảng, mọi cách đều khớp


In [26]:
# Test ngẫu nhiên có seed, đúng ràng buộc: n lẻ <= 10^5, giá trị trong [0, 10^5]
rng = random.Random(540)

def random_case(rng, k_pairs):
    values = sorted(rng.sample(range(0, 100001), k_pairs + 1))   # k+1 giá trị khác nhau, đã tăng dần
    single = rng.choice(values)
    arr = []
    for v in values:
        arr += [v] if v == single else [v, v]
    return arr, single

for _ in range(300):
    k = rng.randint(0, 40)
    arr, single = random_case(rng, k)
    assert len(arr) == 2 * k + 1
    for name, fn in ALL_METHODS.items():
        assert fn(arr) == single, (name, arr)

# Một số mảng lớn
for k in [1000, 30000, 49999]:
    arr, single = random_case(rng, k)
    assert len(arr) <= 10**5
    for name in ["pairs", "xor", "bs_pair_index", "bs_mid_xor", "bs_manual", "leetcode", "counter"]:
        assert ALL_METHODS[name](arr) == single
print("ngẫu nhiên: OK")

# Biên lớn nhất: n = 99999, lẻ loi ở mọi vị trí chọn lọc
for sp in [0, 1, 25000, 49998, 49999]:
    arr = build_with_single_at(49999, sp)
    assert len(arr) == 99999
    assert Solution().singleNonDuplicate(arr) == single_xor(arr)
print("n = 99999: OK")

ngẫu nhiên: OK

n = 99999: OK

## 13. Các lỗi người mới hay gặp

### Lỗi 1 — `r = mid - 1` làm mất đáp án

Với `mid` thỏa `nums[mid] != nums[mid ^ 1]`, `mid` có thể **chính là đáp án** (khi `mid = s`). Nếu đặt `r = mid - 1` ta loại nó ra khỏi khoảng tìm.

In [27]:
def wrong_r_minus_1(nums):
    l, r = 0, len(nums) - 1
    while l < r:
        mid = (l + r) >> 1
        if nums[mid] != nums[mid ^ 1]:
            r = mid - 1          # SAI: có thể bỏ mất chính đáp án
        else:
            l = mid + 1
    return nums[l]

bad = [1, 1, 2, 3, 3]
print("sai:", wrong_r_minus_1(bad), "| đúng:", Solution().singleNonDuplicate(bad))

# Đúng: giữ mid lại
def right_keep_mid(nums):
    l, r = 0, len(nums) - 1
    while l < r:
        mid = (l + r) >> 1
        if nums[mid] != nums[mid ^ 1]:
            r = mid
        else:
            l = mid + 1
    return nums[l]
print("đúng:", right_keep_mid(bad))

sai: 1 | đúng: 2
đúng: 2


### Lỗi 2 — Quên xét chẵn/lẻ (đã thấy ở mục 12)

`nums[mid] != nums[mid + 1]` chỉ đúng khi `mid` chẵn. Với `mid` lẻ thì phải so với `mid - 1`. Dùng `mid ^ 1` (hoặc `partner` tường minh) giải quyết gọn. Ví dụ đã chạy: `wrong_no_parity` trả sai ở cả hai ví dụ đề.

### Lỗi 3 — `while l <= r` kèm `r = mid` gây vòng lặp vô hạn

Khi `l == r`, `mid = l`; nếu nhánh `r = mid` được chọn thì `r` không đổi và vòng lặp không bao giờ kết thúc. Quy tắc: **`r = mid` đi với `while l < r`**; `while l <= r` đi với `r = mid - 1`. Ta mô phỏng với "bộ đếm an toàn" để không treo máy.

In [28]:
def wrong_infinite(nums, limit=50):
    l, r = 0, len(nums) - 1
    steps = 0
    while l <= r:                      # SAI khi đi kèm r = mid
        steps += 1
        if steps > limit:
            return "VÒNG LẶP VÔ HẠN (đã dừng sau %d bước)" % limit
        mid = (l + r) >> 1
        if nums[mid] != nums[mid ^ 1]:
            r = mid
        else:
            l = mid + 1
    return nums[l]

print(wrong_infinite([1, 1, 2, 3, 3, 4, 4, 8, 8]))

VÒNG LẶP VÔ HẠN (đã dừng sau 50 bước)


### Lỗi 4 — Trả về chỉ số thay vì giá trị

Đề hỏi **giá trị** của phần tử lẻ loi. `return l` trả về chỉ số. Hãy luôn kiểm tra ví dụ: ví dụ 1 có đáp án `2` ở chỉ số `2` — trùng hợp! Ví dụ 2 (đáp án `10` ở chỉ số `4`) mới lộ ra lỗi. Bài học: đừng chỉ thử một ví dụ.

In [29]:
def wrong_return_index(nums):
    l, r = 0, len(nums) - 1
    while l < r:
        mid = (l + r) >> 1
        if nums[mid] != nums[mid ^ 1]:
            r = mid
        else:
            l = mid + 1
    return l                        # SAI: trả chỉ số

print(wrong_return_index([1, 1, 2, 3, 3, 4, 4, 8, 8]), "(trùng hợp đúng)")
print(wrong_return_index([3, 3, 7, 7, 10, 11, 11]), "(đáng lẽ 10)")

2 (trùng hợp đúng)
4 (đáng lẽ 10)


### Lỗi 5 — Lo `mid ^ 1` vượt mảng rồi thêm kiểm tra thừa (hoặc bỏ sót chỗ thật sự nguy hiểm)

Trong khuôn mẫu `while l < r`, `mid ^ 1` luôn hợp lệ (đã chứng minh ở mục 10), nên **không** cần `if mid + 1 < n`. Nhưng nếu bạn đổi khuôn mẫu thành `r = n` (nửa mở) hoặc `while l <= r`, `mid` có thể bằng `n - 1` và `mid ^ 1 = n` gây `IndexError`. Hãy giữ đúng khuôn mẫu.

In [30]:
def wrong_index_error(nums):
    l, r = 0, len(nums) - 1
    while l <= r:                         # đổi khuôn mẫu: có thể chạm mid = n - 1
        mid = (l + r) >> 1
        if nums[mid] != nums[mid ^ 1]:    # mid = n - 1 (chẵn) -> mid ^ 1 = n: vượt mảng
            r = mid - 1
        else:
            l = mid + 1
    return nums[l]

try:
    print(wrong_index_error([1, 1, 2, 2, 9]))
except IndexError as e:
    print("IndexError:", e)

IndexError: list index out of range


## 14. Mẹo & nhận dạng pattern

- **"Mảng đã sắp xếp" + yêu cầu `O(log n)`** → gần như chắc chắn là binary search. Câu hỏi quan trọng: *vị từ đơn điệu ở đây là gì?*
- **Dữ liệu đi theo cặp, trừ một cái lẻ loi:** nghĩ ngay tới **chẵn/lẻ của chỉ số** — vị trí của cặp bị "lệch pha" sau điểm lẻ loi.
- **`i ^ 1`** = bạn cặp của `i` trong cách ghép `(0,1), (2,3), ...`. Tương tự, `i - i % k` cho đầu nhóm khi mỗi nhóm có `k` phần tử.
- Nếu **không sắp xếp** → XOR toàn mảng (`O(n)`, `O(1)`) là thủ thuật kinh điển (Single Number). Nếu sắp xếp mà không bị yêu cầu `O(log n)` → duyệt theo cặp là đủ.
- Khi viết binary search tìm "vị trí đầu tiên đúng": `while l < r`, `r = mid` khi đúng, `l = mid + 1` khi sai, trả `l`. Luôn tự hỏi: `mid` có thể là đáp án không? (có → giữ `mid`).
- Kiểm tra biên trước khi nộp: `n = 1`, lẻ loi ở đầu, lẻ loi ở cuối.

## 15. Biến thể

### 15.1. Trả về chỉ số thay vì giá trị

Chỉ cần `return l` ở cuối (nhưng phải đọc kỹ đề đòi gì!).

### 15.2. Mảng không sắp xếp

Binary search không còn dùng được. Dùng XOR (`O(n)`, `O(1)`) hoặc `Counter` (`O(n)`, `O(n)`). Đây là *Single Number* (LeetCode 136).

### 15.3. Mỗi giá trị xuất hiện 3 lần, trừ một giá trị xuất hiện 1 lần (mảng đã sắp xếp)

Các nhóm 3 nguyên vẹn nằm ở chỉ số `(0,1,2), (3,4,5), ...`. Với `mid`, đầu nhóm là `g = mid - mid % 3`. Nếu `nums[g] == nums[g + 2]` thì nhóm nguyên vẹn → lẻ loi ở bên phải; ngược lại lẻ loi ở `mid` hoặc bên trái. Đây là Bài 5 phần luyện tập; cell dưới là bản mẫu để bạn đối chiếu sau khi tự làm.

In [31]:
def single_in_triples(nums):
    """Sorted, mỗi giá trị xuất hiện đúng 3 lần trừ một giá trị xuất hiện 1 lần."""
    l, r = 0, len(nums) - 1
    while l < r:
        mid = (l + r) >> 1
        g = mid - mid % 3                         # đầu nhóm 3 chứa mid
        if g + 2 < len(nums) and nums[g] == nums[g + 2]:
            l = mid + 1                           # nhóm nguyên vẹn -> lẻ loi bên phải
        else:
            r = mid
    return nums[l]

def build_triples(k_groups, s_group):
    arr, v = [], 0
    for idx in range(k_groups + 1):
        if idx == s_group:
            arr.append(v); v += 1
        if idx < k_groups:
            arr += [v] * 3; v += 1
    return arr

for k in range(0, 10):
    for sp in range(k + 1):
        arr = build_triples(k, sp)
        assert single_in_triples(arr) == [x for x, c in Counter(arr).items() if c == 1][0]
print(single_in_triples([1, 1, 1, 2, 3, 3, 3]), single_in_triples([4]), single_in_triples([1, 2, 2, 2]))

2

 4 1


### 15.4. Vị từ khác, cùng khuôn mẫu

Các bài "tìm vị trí đầu tiên mà điều kiện đúng" khác: *Find Peak Element* (162), *Find Minimum in Rotated Sorted Array* (153), *Search Insert Position* (35). Điểm chung: tìm được vị từ đơn điệu rồi dùng `while l < r`.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Counter | Đếm tần suất | `O(n)` | `O(n)` | Mảng bất kỳ, chỉ cần đúng, không sợ tốn bộ nhớ |
| 2. Duyệt theo cặp | Cặp đầu tiên `nums[i] != nums[i+1]` (`i` chẵn) | `O(n)` | `O(1)` | Mảng sắp xếp, không yêu cầu `O(log n)` |
| 3. XOR | Các cặp triệt tiêu | `O(n)` | `O(1)` | Mảng **không** sắp xếp, ngắn gọn |
| 4. BS chỉ số cặp | Tìm cặp lệch đầu tiên | `O(log n)` | `O(1)` | Khi thích nghĩ theo cặp, dễ hình dung |
| 5. BS `mid ^ 1` | Vị từ `nums[i] != nums[i^1]` đơn điệu | `O(log n)` | `O(1)` | **Nộp bài**, ngắn nhất, đạt yêu cầu đề |

In [32]:
arr = build_with_single_at(49999, 40000)     # n = 99999, lẻ loi ở gần cuối
for name in ["counter", "pairs", "xor", "bs_pair_index", "bs_mid_xor"]:
    fn = ALL_METHODS[name]
    print(f"{name:14s} đáp án {fn(arr)}")
    assert fn(arr) == single_xor(arr)

counter        đáp án 40000


pairs          đáp án 40000
xor            đáp án 40000
bs_pair_index  đáp án 40000
bs_mid_xor     đáp án 40000


## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output của ba dòng sau, rồi chạy để kiểm tra:

```python
print([i ^ 1 for i in range(6)])
print([(i, i ^ 1) for i in (6, 7, 10)])
print(single_bs_xor([2, 2, 5, 5, 6, 9, 9]))
```

In [33]:
# Bài 1: viết dự đoán của bạn vào biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay binary search (dễ-vừa)

Với `nums = [2,2,5,5,6,9,9]`, hãy lập bảng các bước `l, r, mid, mid ^ 1, nums[mid], nums[mid ^ 1]` và hành động. Hỏi: sau bao nhiêu bước dừng, và đáp án là gì?

In [34]:
# Bài 2: điền kết quả tính tay
hand_steps = None     # số bước lặp
hand_answer = None    # đáp án

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành hàm theo khuôn mẫu `while l < r`.

In [35]:
def exercise_3_single(nums: List[int]) -> int:
    l, r = 0, len(nums) - 1
    # TODO 1: lặp khi l < r
    #     TODO 2: mid = trung điểm làm tròn xuống
    #     TODO 3: nếu nums[mid] != nums[mid ^ 1] thì thu hẹp về mid, ngược lại bỏ mid sang phải
    # TODO 4: trả về GIÁ TRỊ (không phải chỉ số)
    return None


# assert exercise_3_single([1, 1, 2, 3, 3, 4, 4, 8, 8]) == 2
# assert exercise_3_single([3, 3, 7, 7, 10, 11, 11]) == 10
# assert exercise_3_single([7]) == 7

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại cách 4, hãy tự viết `exercise_4_pair_index(nums)`: binary search trên **chỉ số cặp** `k` (so `nums[2k]` với `nums[2k+1]`).

In [36]:
def exercise_4_pair_index(nums: List[int]) -> int:
    # TODO
    pass


# assert exercise_4_pair_index([1, 1, 2, 3, 3, 4, 4, 8, 8]) == 2
# assert exercise_4_pair_index([1, 2, 2]) == 1
# assert exercise_4_pair_index([4, 4, 9]) == 9

### Bài 5 — Mini challenge: mỗi giá trị xuất hiện 3 lần (khó)

Mảng đã sắp xếp; mọi giá trị xuất hiện **đúng 3 lần, trừ một giá trị xuất hiện đúng 1 lần**. Viết `exercise_5_triples(nums)` trả về giá trị đó trong `O(log n)`. Gợi ý: nhóm `(0,1,2), (3,4,5), ...`; đầu nhóm là `mid - mid % 3`.

```python
exercise_5_triples([1, 1, 1, 2, 3, 3, 3]) -> 2
exercise_5_triples([4]) -> 4
exercise_5_triples([1, 2, 2, 2]) -> 1
```

In [37]:
def exercise_5_triples(nums: List[int]) -> int:
    # TODO
    pass


# assert exercise_5_triples([1, 1, 1, 2, 3, 3, 3]) == 2
# assert exercise_5_triples([4]) == 4
# assert exercise_5_triples([1, 2, 2, 2]) == 1

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `i ^ 1` lật bit thấp nhất của `i`.
- Mức 2: `6 <-> 7`, `10 <-> 11`; dòng 3 chạy binary search với `n = 7`.
- Mức 3: dòng 1 là `[1, 0, 3, 2, 5, 4]`; dòng 3 là giá trị lẻ loi (cặp `2,2` `5,5` rồi `6`).

**Bài 2**
- Mức 1: bắt đầu `l = 0`, `r = 6`, `mid = 3`.
- Mức 2: `nums[3] = 5`, `3 ^ 1 = 2`, `nums[2] = 5` → bằng nhau → `l = 4`.
- Mức 3: tiếp theo `mid = 5` (khác) → `r = 5`, rồi `mid = 4` (khác) → `r = 4`. Tổng cộng 3 bước.

**Bài 3**
- Mức 1: giống khuôn mẫu mục 3.3.
- Mức 2: `mid = (l + r) >> 1`; khác nhau thì `r = mid`, bằng nhau thì `l = mid + 1`.
- Mức 3: sau vòng lặp `return nums[l]`.

**Bài 4**
- Mức 1: `lo, hi = 0, (n - 1) // 2`.
- Mức 2: với `k = (lo + hi) >> 1`, nếu `nums[2k] != nums[2k+1]` thì `hi = k`, ngược lại `lo = k + 1`.
- Mức 3: kết quả `nums[2 * lo]`; để ý `n = 1` thì `hi = 0` nên không vào vòng lặp.

**Bài 5**
- Mức 1: lẻ loi luôn ở chỉ số chia hết cho 3 (trước nó là các nhóm 3 đầy đủ).
- Mức 2: `g = mid - mid % 3`; nhóm nguyên vẹn khi `nums[g] == nums[g + 2]`.
- Mức 3: nhóm nguyên vẹn → `l = mid + 1`; ngược lại `r = mid`. Trả `nums[l]`. Cẩn thận `g + 2` có thể vượt mảng khi `n` nhỏ (kiểm tra `g + 2 < n`).

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [38]:
print([i ^ 1 for i in range(6)])
print([(i, i ^ 1) for i in (6, 7, 10)])
print(single_bs_xor([2, 2, 5, 5, 6, 9, 9]))

assert [i ^ 1 for i in range(6)] == [1, 0, 3, 2, 5, 4]
assert [(i, i ^ 1) for i in (6, 7, 10)] == [(6, 7), (7, 6), (10, 11)]
assert single_bs_xor([2, 2, 5, 5, 6, 9, 9]) == 6

[1, 0, 3, 2, 5, 4]
[(6, 7), (7, 6), (10, 11)]
6


### Đáp án Bài 2

| bước | l | r | mid | mid ^ 1 | nums[mid] | nums[mid^1] | hành động |
|---|---|---|---|---|---|---|---|
| 1 | 0 | 6 | 3 | 2 | 5 | 5 | bằng, `l = 4` |
| 2 | 4 | 6 | 5 | 4 | 9 | 6 | khác, `r = 5` |
| 3 | 4 | 5 | 4 | 5 | 6 | 9 | khác, `r = 4` |

Dừng sau **3 bước**, đáp án `6`.

In [39]:
ans_2 = trace_single([2, 2, 5, 5, 6, 9, 9])
assert ans_2 == 6

nums = [2, 2, 5, 5, 6, 9, 9]
bước 1: l=0 r=6 mid=3 mid^1=2 nums[mid]=5 nums[mid^1]=5 -> bằng -> l = 4
bước 2: l=4 r=6 mid=5 mid^1=4 nums[mid]=9 nums[mid^1]=6 -> khác -> r = 5
bước 3: l=4 r=5 mid=4 mid^1=5 nums[mid]=6 nums[mid^1]=9 -> khác -> r = 4
dừng: l=r=4 -> đáp án 6


### Đáp án Bài 3

In [40]:
def exercise_3_single_solution(nums: List[int]) -> int:
    l, r = 0, len(nums) - 1
    while l < r:
        mid = (l + r) >> 1
        if nums[mid] != nums[mid ^ 1]:
            r = mid
        else:
            l = mid + 1
    return nums[l]

assert exercise_3_single_solution([1, 1, 2, 3, 3, 4, 4, 8, 8]) == 2
assert exercise_3_single_solution([3, 3, 7, 7, 10, 11, 11]) == 10
assert exercise_3_single_solution([7]) == 7
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [41]:
def exercise_4_pair_index_solution(nums: List[int]) -> int:
    lo, hi = 0, (len(nums) - 1) // 2
    while lo < hi:
        k = (lo + hi) >> 1
        if nums[2 * k] != nums[2 * k + 1]:
            hi = k
        else:
            lo = k + 1
    return nums[2 * lo]

assert exercise_4_pair_index_solution([1, 1, 2, 3, 3, 4, 4, 8, 8]) == 2
assert exercise_4_pair_index_solution([1, 2, 2]) == 1
assert exercise_4_pair_index_solution([4, 4, 9]) == 9
for k in range(0, 10):
    for sp in range(k + 1):
        a = build_with_single_at(k, sp)
        assert exercise_4_pair_index_solution(a) == single_xor(a)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [42]:
def exercise_5_triples_solution(nums: List[int]) -> int:
    l, r = 0, len(nums) - 1
    while l < r:
        mid = (l + r) >> 1
        g = mid - mid % 3
        if g + 2 < len(nums) and nums[g] == nums[g + 2]:
            l = mid + 1
        else:
            r = mid
    return nums[l]

assert exercise_5_triples_solution([1, 1, 1, 2, 3, 3, 3]) == 2
assert exercise_5_triples_solution([4]) == 4
assert exercise_5_triples_solution([1, 2, 2, 2]) == 1
for k in range(0, 12):
    for sp in range(k + 1):
        a = build_triples(k, sp)
        expected = [x for x, c in Counter(a).items() if c == 1][0]
        assert exercise_5_triples_solution(a) == expected, a
print("Bài 5 OK")

Bài 5 OK

## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
class Solution:
    def singleNonDuplicate(self, nums: List[int]) -> int:
        l, r = 0, len(nums) - 1
        while l < r:
            mid = (l + r) >> 1
            if nums[mid] != nums[mid ^ 1]:   # mid đã lệch cặp: lẻ loi ở mid hoặc bên trái
                r = mid
            else:                            # mid còn trong cặp (chẵn, lẻ): lẻ loi bên phải
                l = mid + 1
        return nums[l]
```

Câu thần chú: **"chẵn so với phải, lẻ so với trái = so với `mid ^ 1`; khác nhau thì `r = mid`, bằng nhau thì `l = mid + 1`."**

### Checklist trước khi submit

- [ ] `while l < r` đi cùng `r = mid` và `l = mid + 1` (không dùng `mid - 1`).
- [ ] So với `mid ^ 1`, không phải `mid + 1`.
- [ ] Trả về `nums[l]` (giá trị), không phải `l`.
- [ ] Test `n = 1`, lẻ loi ở đầu, lẻ loi ở cuối, lẻ loi ở giữa.
- [ ] Đúng yêu cầu `O(log n)` thời gian, `O(1)` bộ nhớ (không `Counter`, không duyệt toàn mảng).
- [ ] Skeleton đúng: `class Solution`, method `singleNonDuplicate(self, nums: List[int]) -> int`.